# `apptbl-magistro-subsets`: AI & offshoring OLS treatment-effect error by country, arm, and attribute type

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSET_NAME = "apptbl-magistro-subsets"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
if not (PAPER_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from manuscript-training-assets/; {PAPER_DIR} has no microdata_analysis.py")
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/apptbl-magistro-subsets.*


## Run design

In [2]:
RUN_ROOTS = (PAPER_DIR / "runs-rorqual", PAPER_DIR / "runs-nibi")
MODELS = ("kev-0.8b", "kev-4b", "kev-9b")
BASE_SLUG = "row1_base_untrained"
TRAINED_SLUGS = ("row2b_wu_ce_oracle", "row3_berke_ce_only", "row5_full_berke_wu_ce_oracle")
VALIDATORS = ("kriner2026", "grummon2025")

ctx = ma.load_microdata_context(
    run_roots=RUN_ROOTS,
    models=MODELS,
    base_slug=BASE_SLUG,
    trained_slugs=TRAINED_SLUGS,
    validators=VALIDATORS,
)
display(ctx.status_grid)

,kev-0.8b,kev-4b,kev-9b
regime,,,
row1_base_untrained,step 0 [test_best ✓],step 0 [test_best ✓],step 0 [test_best ✓]
row2b_wu_ce_oracle,finished (@640/1395) [test_best ✓],finished (@416/1395) [test_best ✓],finished (@928/1395) [test_best ✓]
row3_berke_ce_only,finished (@1152/2730) [test_best ✓],finished (@544/2730) [test_best ✓],finished (@256/2730) [test_best ✓]
row5_full_berke_wu_ce_oracle,finished (@1600/4125) [test_best ✓],finished (@1472/4125) [test_best ✓],finished (@1600/4125) [test_best ✓]


## Labels and formatting (edit wording here)

In [3]:
MODEL_LABELS = {"kev-0.8b": "KEV-0.8B", "kev-4b": "KEV-4B", "kev-9b": "KEV-9B"}

# Row order follows this dict.
REGIME_LABELS = {
    BASE_SLUG: "Off-the-shelf",
    "row2b_wu_ce_oracle": "Electricity",
    "row3_berke_ce_only": "Drones",
    "row5_full_berke_wu_ce_oracle": "Drones + Electricity",
}

# Each model gets its own row spanning the full table, above its regime rows.
MODEL_ROW_TEMPLATE = "*{model_label}*"
STUB_HEADERS = ("Trained on",)

US_REGRESSIONS = ("magistro2025/us_ai", "magistro2025/us_offshoring")
CANADA_REGRESSIONS = ("magistro2025/ca_ai", "magistro2025/ca_offshoring")
ALL_REGRESSIONS = US_REGRESSIONS + CANADA_REGRESSIONS

# Column label -> (regressions to include, which terms: "all", "price", "non-price"). {n} becomes the number of slopes.
SUBSET_COLUMNS = {
    "All ({n})": (ALL_REGRESSIONS, "all"),
    "US ({n})": (US_REGRESSIONS, "all"),
    "Canada ({n})": (CANADA_REGRESSIONS, "all"),
    "US AI ({n})": (("magistro2025/us_ai",), "all"),
    "US Off. ({n})": (("magistro2025/us_offshoring",), "all"),
    "CA AI ({n})": (("magistro2025/ca_ai",), "all"),
    "CA Off. ({n})": (("magistro2025/ca_offshoring",), "all"),
    "Price ({n})": (ALL_REGRESSIONS, "price"),
    "Jobs ({n})": (ALL_REGRESSIONS, "non-price"),
}
PRICE_MARKER = "| Price:"

MSE_SCALE = 1000.0
MSE_DECIMALS = 2

## Build the table

In [4]:
import math
import re

from matplotlib import font_manager
from PIL import ImageFont

# Page and table style of manuscript/quarto-files/ms_template.docx. Word fixes the column widths that Pandoc
# derives from the rule dash counts, so a column narrower than its longest word splits that word across lines.
TEXT_WIDTH_PT = 468.0  # 8.5 in page minus 1 in left and right margins
TABLE_FONT_PT = 10.0  # Compact paragraph style used in table cells
CELL_PADDING_PT = 10.8  # Table style left + right cell margins (108 twips each)
COLUMN_SLACK_PT = 2.0  # per-column allowance for differences between Word and the font metrics measured here
MATH_WIDTH_FACTOR = 1.15  # Word sets $...$ in Cambria Math, which runs wider than Times New Roman
# No-break spaces around ×: Word sets each $...$ as one equation that never wraps.
MATH_SYMBOLS = {r"\beta": "β", r"\rho": "ρ", r"\times": "\u00a0×\u00a0"}


def load_table_font(weight):
    font_properties = font_manager.FontProperties(family="Times New Roman", weight=weight)
    font_path = font_manager.findfont(font_properties, fallback_to_default=False)
    return ImageFont.truetype(font_path, 1000)


REGULAR_FONT = load_table_font("normal")
BOLD_FONT = load_table_font("bold")


def shown_math(match):
    latex = match.group(1)
    text = latex
    for command, symbol in MATH_SYMBOLS.items():
        text = text.replace(command, symbol)
    text = text.replace(r"\mathrm", "")
    for mark in ("{", "}", "_", "^"):
        text = text.replace(mark, "")
    if "\\" in text:
        raise ValueError(f"No display text for a LaTeX command in {latex!r}; add it to MATH_SYMBOLS")
    return text


def shown_citation(match):
    surname = match.group(1).title()
    year = match.group(2)
    return f"({surname} {year})"


def shown_cell(cell):
    """Text Word displays for a markdown cell, with the font and width factor to measure it."""
    is_bold = cell.startswith("**") and cell.endswith("**")
    text = cell
    if is_bold:
        text = text[2:-2]
    text = re.sub(r"\[@([a-z]+)(\d{4})[a-z0-9]*\]", shown_citation, text)
    has_math = re.search(r"(?<!\\)\$", text) is not None
    text = re.sub(r"(?<!\\)\$(.+?)(?<!\\)\$", shown_math, text)
    text = text.replace(r"\$", "$")
    text = text.replace(r"\|", "|")
    font = BOLD_FONT if is_bold else REGULAR_FONT
    factor = MATH_WIDTH_FACTOR if has_math else 1.0
    return text, font, factor


def text_width_pt(text, font, factor):
    return font.getlength(text) / 1000.0 * TABLE_FONT_PT * factor


def unbreakable_pieces(text):
    """Pieces Word keeps on one line: words, with compound words (letter-hyphen-letter) split after the hyphen."""
    pieces = []
    for word in re.split(r"[ \t\n]+", text):
        pieces.extend(re.split(r"(?<=[A-Za-z]-)(?=[A-Za-z])", word))
    return pieces


def column_widths_pt(headers, rows):
    """Each column gets at least its widest unbreakable piece; leftover page width goes to columns that wrap."""
    n_columns = len(headers)
    min_pt = [0.0] * n_columns
    natural_pt = [0.0] * n_columns
    for cells in [headers, *rows]:
        if len(cells) != n_columns:
            raise ValueError(f"Row has {len(cells)} cells, expected {n_columns}: {cells}")
        for column_index, cell in enumerate(cells):
            text, font, factor = shown_cell(cell)
            for piece in unbreakable_pieces(text):
                piece_pt = text_width_pt(piece, font, factor)
                min_pt[column_index] = max(min_pt[column_index], piece_pt)
            cell_pt = text_width_pt(text, font, factor)
            natural_pt[column_index] = max(natural_pt[column_index], cell_pt)
    extra_pt = CELL_PADDING_PT + COLUMN_SLACK_PT
    min_pt = [width + extra_pt for width in min_pt]
    natural_pt = [width + extra_pt for width in natural_pt]
    if sum(min_pt) > TEXT_WIDTH_PT:
        raise ValueError(
            f"Longest words need {sum(min_pt):.0f}pt but the page is {TEXT_WIDTH_PT:.0f}pt wide; "
            f"shorten labels or drop columns. Per-column minimum: {[round(width) for width in min_pt]}"
        )
    if sum(natural_pt) <= TEXT_WIDTH_PT:
        return natural_pt
    spare_pt = TEXT_WIDTH_PT - sum(min_pt)
    stretch_pt = [natural - minimum for natural, minimum in zip(natural_pt, min_pt)]
    total_stretch_pt = sum(stretch_pt)
    widths_pt = []
    for minimum, stretch in zip(min_pt, stretch_pt):
        widths_pt.append(minimum + spare_pt * stretch / total_stretch_pt)
    return widths_pt


def grid_table(headers, table_body, column_rules):
    """Pandoc grid table. table_body rows: ("model", text) spans every column; ("data", cells) has one cell per column.

    column_rules set alignment (':---' left, '---:' right); column widths come from column_widths_pt.
    """
    n_columns = len(headers)
    if len(column_rules) != n_columns:
        raise ValueError(f"{len(column_rules)} column rules for {n_columns} columns")
    data_rows = []
    for row_kind, row_content in table_body:
        if row_kind == "data":
            data_rows.append(row_content)
        elif row_kind != "model":
            raise ValueError(f"Unknown body row kind {row_kind!r}")
    widths_pt = column_widths_pt(headers, data_rows)

    # Grid cells hold their raw markdown, so convert the point widths to characters with the smallest
    # factor that fits every cell; the proportions between columns stay the same.
    chars_per_pt = 0.0
    for column_index in range(n_columns):
        widest_raw = len(headers[column_index])
        for row in data_rows:
            widest_raw = max(widest_raw, len(row[column_index]))
        chars_per_pt = max(chars_per_pt, widest_raw / widths_pt[column_index])
    widths = []
    for width_pt in widths_pt:
        widths.append(math.ceil(width_pt * chars_per_pt))

    def padded(text, width):
        return " " + text.ljust(width) + " "

    row_rule = "+" + "+".join("-" * (width + 2) for width in widths) + "+"
    header_end_parts = []
    for column_rule, width in zip(column_rules, widths):
        if re.fullmatch(r":?-+:?", column_rule) is None:
            raise ValueError(f"Bad column rule {column_rule!r}")
        if column_rule.endswith(":"):
            header_end_parts.append("=" * (width + 1) + ":")
        else:
            header_end_parts.append(":" + "=" * (width + 1))
    header_end = "+" + "+".join(header_end_parts) + "+"
    # A model row is one cell spanning every column: no inner | on its line, column + marks on the rules around it.
    full_inner_width = sum(widths) + 3 * (n_columns - 1)

    lines = [row_rule]
    lines.append("|" + "|".join(padded(headers[c], widths[c]) for c in range(n_columns)) + "|")
    lines.append(header_end)
    for row_kind, row_content in table_body:
        if row_kind == "model":
            lines.append("|" + padded(row_content, full_inner_width) + "|")
        else:
            lines.append("|" + "|".join(padded(row_content[c], widths[c]) for c in range(n_columns)) + "|")
        lines.append(row_rule)
    return "\n".join(lines) + "\n"

In [5]:
def subset_mask(betas, regressions, term_group):
    in_regressions = betas["model"].isin(regressions)
    is_price = betas.index.str.contains(PRICE_MARKER, regex=False)
    if term_group == "all":
        return in_regressions
    if term_group == "price":
        return in_regressions & is_price
    if term_group == "non-price":
        return in_regressions & ~is_price
    raise ValueError(f"Unknown term group {term_group!r}")


reference_betas = ctx.magistro_betas_by_model[MODELS[0]][BASE_SLUG]
column_headers = []
for column_template, (regressions, term_group) in SUBSET_COLUMNS.items():
    mask = subset_mask(reference_betas, regressions, term_group)
    if not mask.any():
        raise ValueError(f"Empty subset for column {column_template!r}")
    column_headers.append(column_template.replace("{n}", str(int(mask.sum()))))

# Bold marks the lowest value per model and subset.
# table_body lists the body rows in order: ("model", text) for a model row, ("data", cells) for a regime row.
table_body = []
display_rows = []
for model_slug in MODELS:
    mse_by_regime = {}
    for regime_slug in REGIME_LABELS:
        betas = ctx.magistro_betas_by_model[model_slug][regime_slug]
        squared_errors = (betas["model_pop_beta"] - betas["human_pop_beta"]) ** 2
        subset_mses = []
        for regressions, term_group in SUBSET_COLUMNS.values():
            mask = subset_mask(betas, regressions, term_group)
            subset_mses.append(float(squared_errors.loc[mask].mean()))
        mse_by_regime[regime_slug] = subset_mses
    mse_df = pd.DataFrame(mse_by_regime, index=column_headers).T
    best_slugs = mse_df.idxmin()

    model_row_text = MODEL_ROW_TEMPLATE.format(model_label=MODEL_LABELS[model_slug])
    table_body.append(("model", model_row_text))
    for regime_slug, regime_label in REGIME_LABELS.items():
        row = [regime_label]
        for column_header in column_headers:
            text = f"{mse_df.loc[regime_slug, column_header] * MSE_SCALE:.{MSE_DECIMALS}f}"
            if best_slugs[column_header] == regime_slug:
                text = f"**{text}**"
            row.append(text)
        table_body.append(("data", row))
        display_rows.append([MODEL_LABELS[model_slug], *row])

headers = [*STUB_HEADERS, *column_headers]
column_rules = [":---"] * len(STUB_HEADERS) + ["---:"] * len(column_headers)
table_md = grid_table(headers, table_body, column_rules)
display(pd.DataFrame(display_rows, columns=["Model", *headers]))

,Model,Trained on,All (32),US (16),Canada (16),US AI (8),US Off. (8),CA AI (8),CA Off. (8),Price (8),Jobs (24)
0,KEV-0.8B,Off-the-shelf,33.92,41.48,26.35,50.62,32.34,24.20,28.51,77.60,19.36
1,KEV-0.8B,Electricity,29.60,35.59,23.62,42.21,28.96,20.16,27.08,71.71,15.57
2,KEV-0.8B,Drones,34.41,41.14,27.67,49.82,32.46,25.25,30.10,83.27,18.12
3,KEV-0.8B,Drones + Electricity,**26.61**,**32.17**,**21.06**,**38.38**,**25.96**,**17.60**,**24.51**,**67.37**,**13.03**
4,KEV-4B,Off-the-shelf,99.01,93.76,104.27,92.30,95.22,96.88,111.65,16.09,126.66
5,KEV-4B,Electricity,113.26,106.45,120.08,101.87,111.03,112.91,127.26,11.67,147.13
6,KEV-4B,Drones,74.69,67.29,82.09,60.96,73.62,74.73,89.45,6.21,97.52
7,KEV-4B,Drones + Electricity,**63.98**,**56.05**,**71.90**,**53.45**,**58.66**,**70.30**,**73.49**,**5.66**,**83.42**
8,KEV-9B,Off-the-shelf,82.89,75.34,90.43,67.71,82.98,80.55,100.31,**21.31**,103.41
9,KEV-9B,Electricity,95.68,83.68,107.69,71.35,96.01,98.52,116.86,37.05,115.23


## Save `apptbl-magistro-subsets.md` (included by the manuscript)

In [6]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

+----------------------+---------------+---------------+-----------------+---------------+----------------+---------------+----------------+--------------+---------------+
| Trained on           | All (32)      | US (16)       | Canada (16)     | US AI (8)     | US Off. (8)    | CA AI (8)     | CA Off. (8)    | Price (8)    | Jobs (24)     |
+:=====================+==============:+==============:+================:+==============:+===============:+==============:+===============:+=============:+==============:+
| *KEV-0.8B*                                                                                                                                                              |
+----------------------+---------------+---------------+-----------------+---------------+----------------+---------------+----------------+--------------+---------------+
| Off-the-shelf        | 33.92         | 41.48         | 26.35           | 50.62         | 32.34          | 24.20         | 28.51          |